# Gradient à pas optimal

**Principe** : même direction que le gradient simple, d_k = −∇f(x_k), mais à chaque itération on choisit le pas qui minimise f le long de cette direction :

α_k = argmin f(x_k + α d_k)

Pour une fonction quadratique, il a une formule fermée : α_k = (g_kᵀ g_k) / (g_kᵀ A g_k), avec g_k = ∇f(x_k).

**Problème test** : f(x) = ½ xᵀAx − bᵀx avec A = [[6, 2], [2, 3]], b = [2, 1], x₀ = [3, −2].
Le minimiseur exact est x* = [2/7, 1/7].

**Exemple à la main** :
- g₀ = Ax₀ − b = [12, −1]
- g₀ᵀg₀ = 145 et A g₀ = [70, 21], donc g₀ᵀ A g₀ = 819
- α₀ = 145/819 ≈ 0,1770
- x₁ = [3, −2] − 0,1770·[12, −1] ≈ [0,8755 ; −1,8230]

**Propriété** : le nouveau gradient est orthogonal à l'ancien (∇f(x₁)ᵀ∇f(x₀) = 0), donc la trajectoire fait des zigzags à angle droit.

In [3]:
import numpy as np

A = np.array([[6.0, 2.0], [2.0, 3.0]])
b = np.array([2.0, 1.0])
f = lambda x: 0.5 * x @ A @ x - b @ x
grad_f = lambda x: A @ x - b

def gradient_pas_optimal(grad, x0, tol=1e-6, max_iter=10000):
    x = np.array(x0, dtype=float)
    xs = [x.copy()]
    normes = [np.linalg.norm(grad(x))]
    alphas = []
    for k in range(max_iter):
        g = grad(x)
        if np.linalg.norm(g) < tol:
            break
        alpha = (g @ g) / (g @ A @ g)   # pas optimal (cas quadratique)
        x = x - alpha * g
        xs.append(x.copy())
        normes.append(np.linalg.norm(grad(x)))
        alphas.append(alpha)
    return x, xs, normes, alphas

x0 = [3.0, -2.0]
x_star = np.linalg.solve(A, b)
x, xs, normes, alphas = gradient_pas_optimal(grad_f, x0)

print("x =", x, "| itérations =", len(xs) - 1)
print("erreur =", np.linalg.norm(x - x_star))
print("alpha_0 =", alphas[0])
print("x1 =", xs[1])
print("<grad(x1), grad(x0)> =", grad_f(xs[1]) @ grad_f(x0))

x = [0.28571434 0.14285695] | itérations = 25
erreur = 2.003337366660048e-07
alpha_0 = 0.17704517704517705
x1 = [ 0.87545788 -1.82295482]
<grad(x1), grad(x0)> = 8.881784197001252e-16


**Conclusion** : l'algorithme converge en 25 itérations (contre 71 pour le pas fixe), avec une erreur de 2·10⁻⁷. Le produit scalaire des gradients successifs vaut environ 10⁻¹⁵, ce qui confirme l'orthogonalité. La convergence reste linéaire, de taux au pire (κ−1)/(κ+1) ≈ 0,556 avec κ = 7/2.